# 04 · Route selectively with a direct exchange

**Level:** beginner · **Time:** 25–35 minutes · **Prerequisite:** notebook 03.

Build log routing: an audit queue receives `info` and `error`; an alerts queue receives only `error`.
You will distinguish a message's routing key from a queue's binding key and inspect an unroutable publish.

A direct exchange compares the routing key with binding keys for an exact match. It can route one
publication to several queues with matching bindings. This is selective fanout, not necessarily one-to-one.


In [ ]:
from pathlib import Path
import sys
import json
import time
import pika

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "course" / "rabbit.py").is_file()), None)
assert ROOT is not None, "Open Jupyter from the RabbitMQ course folder."
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from course.rabbit import connect, unique_name, wait_for_message


## 1. Create the route map

| Queue | Binding keys | Purpose |
|---|---|---|
| Audit | `info`, `error` | Keep routine and error logs |
| Alerts | `error` | Notify on errors only |

Predict: will `ERROR` match the binding `error`? Routing keys are case-sensitive strings.


In [ ]:
exchange = unique_name("logs.direct")
audit_queue = unique_name("audit")
alerts_queue = unique_name("alerts")
with connect() as connection:
    channel = connection.channel()
    channel.exchange_declare(exchange=exchange, exchange_type="direct")
    for queue in [audit_queue, alerts_queue]:
        channel.queue_declare(queue=queue)
    for severity in ["info", "error"]:
        channel.queue_bind(exchange=exchange, queue=audit_queue, routing_key=severity)
    channel.queue_bind(exchange=exchange, queue=alerts_queue, routing_key="error")
print("Created", exchange)


## 2. Publish two severities

The application chooses routing keys. RabbitMQ does not inspect the JSON `message` text to decide its
severity. Predict the exact payloads in each queue before running the receiving cell.


In [ ]:
with connect() as connection:
    channel = connection.channel()
    for severity, message in [("info", "order accepted"), ("error", "payment declined")]:
        channel.basic_publish(exchange=exchange, routing_key=severity,
                              body=json.dumps({"severity": severity, "message": message}).encode())
    print("Audit ready:", channel.queue_declare(queue=audit_queue, passive=True).method.message_count)
    print("Alerts ready:", channel.queue_declare(queue=alerts_queue, passive=True).method.message_count)


## 3. Verify routing and acknowledge each delivery

This bounded helper consumes an expected number of deliveries and checks that no extra delivery remains.
It uses this lesson's queue names only. The helper's acknowledgement happens after decoding the JSON.


In [ ]:
def read_expected(channel, queue, count):
    records = []
    for _ in range(count):
        method, _, body = wait_for_message(channel, queue)
        records.append(json.loads(body))
        channel.basic_ack(method.delivery_tag)
    assert channel.basic_get(queue=queue, auto_ack=False)[0] is None
    return records

with connect() as connection:
    channel = connection.channel()
    audit = read_expected(channel, audit_queue, 2)
    alerts = read_expected(channel, alerts_queue, 1)
    assert [entry["severity"] for entry in audit] == ["info", "error"]
    assert [entry["severity"] for entry in alerts] == ["error"]
    print("Audit:", audit)
    print("Alerts:", alerts)


## 4. Publish a key with no binding

A successful socket write does not prove a message reached a queue. For this small experiment, enable
publisher confirms and `mandatory=True`: Pika can then surface an unroutable message as an exception.
Confirms acknowledge broker acceptance, not successful work by a consumer; notebook 07 develops this.


In [ ]:
with connect() as connection:
    channel = connection.channel()
    channel.confirm_delivery()
    try:
        channel.basic_publish(exchange=exchange, routing_key="ERROR", body=b"no exact match",
                              mandatory=True)
    except pika.exceptions.UnroutableError:
        print("Expected: ERROR has no binding; the publisher detected it.")
    else:
        raise AssertionError("Expected the upper-case key to be unroutable.")


## 5. Exercise: make warning logs routable

Bind `warning` to the audit queue, publish a warning, then verify that audit gets it and alerts does not.
Try the solution, then explain how to change it if alerts should also receive warnings.


In [ ]:
with connect() as connection:
    channel = connection.channel()
    channel.queue_bind(exchange=exchange, queue=audit_queue, routing_key="warning")
    channel.basic_publish(exchange=exchange, routing_key="warning",
                          body=json.dumps({"severity": "warning", "message": "stock running low"}).encode())
    assert read_expected(channel, audit_queue, 1)[0]["severity"] == "warning"
    assert read_expected(channel, alerts_queue, 0) == []
    print("Only audit received the warning.")


## 6. Cleanup and checkpoint

To add warning alerts, bind the alerts queue with the `warning` key too. Exchange names, queue names,
and routing keys have different jobs; none has to be identical when using a custom exchange.


In [ ]:
with connect() as connection:
    channel = connection.channel()
    for queue in [audit_queue, alerts_queue]:
        channel.queue_delete(queue=queue)
    channel.exchange_delete(exchange=exchange)
print("Direct-routing resources removed.")


**Troubleshooting:** a direct binding such as `order.*` is a literal string, not a wildcard; use a topic
exchange for patterns. Publishing to an exchange that does not exist closes the channel; that is different
from an existing exchange with no route. Recreate the channel after a broker channel error.

Official reference: [RabbitMQ routing tutorial](https://www.rabbitmq.com/tutorials/tutorial-four-python).
